## Section 1: Load Data & Ground Truth Labels

In [1]:
import pandas as pd
import re

df = pd.read_csv('data/nicar-2026-schedule.csv')
df = df[df['canceled'] != True].copy()

def strip_html(text):
    return re.sub(r'<[^>]+>', '', str(text or '')).strip()

df['description_clean'] = df['description'].apply(strip_html)

# Ground truth: does the track list contain 'AI'?
df['label_ai'] = df['tracks'].apply(
    lambda t: 'AI' in str(t)
)

print(f"Total sessions: {len(df)}")
print(f"AI-tracked:     {df['label_ai'].sum()}")
print(f"Non-AI:         {(~df['label_ai']).sum()}")
df[['session_title', 'tracks', 'label_ai']].head(5)

Total sessions: 231
AI-tracked:     30
Non-AI:         201


,session_title,tracks,label_ai
0,How to get data out of complex documents using AI,"['AI', 'Tools & Tech']",True
1,Everyone can be (and should be) a housing repo...,['Beat reporting'],False
2,AI concepts you should know,['AI'],True
3,From delving into data to door knocking: Findi...,['Story ideas'],False
4,Collaborating in an intergenerational workspac...,['Career'],False


## Section 2: Classify with Local LLM

In [2]:
from openai import OpenAI

client = OpenAI(base_url='http://localhost:1234/v1', api_key='lm-studio')

ALL_TRACKS = [
    'AI', 'Beat reporting', 'Career', 'Data analysis', 'Data viz',
    'Educators', 'Elections', 'Equity, inclusion and accessibility',
    'Event', 'International', 'Local', 'Management', 'Mapping',
    'Public records', 'Research', 'Sports', 'Story ideas',
    'Students', 'Tools & Tech',
]

TRACK_LIST = '\n'.join(f'- {t}' for t in ALL_TRACKS)

def classify_session(title, description, model='meta-llama-3.1-8b-instruct'):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'system',
                'content': (
                    f'You are a conference organizer. Sessions are tagged with one or more of these tracks:\n\n'
                    f'{TRACK_LIST}\n\n'
                    'Answer only "yes" or "no".'
                ),
            },
            {
                'role': 'user',
                'content': (
                    f'Title: {title}\n\n'
                    f'Description: {description}\n\n'
                    'Should this session be tagged with the "AI" track?'
                ),
            },
        ],
        temperature=0,
        max_tokens=500,
    )
    raw = response.choices[0].message.content.strip().lower()
    return 'yes' in raw


In [3]:
from tqdm.notebook import tqdm

predictions = []
for i, row in tqdm(df.iterrows(), total=len(df)):
    pred = classify_session(row['session_title'], row['description_clean'])
    predictions.append(pred)
    label = '✓' if pred == row['label_ai'] else '✗'
    print(f"{label} [{'AI' if pred else 'no':2}] {row['session_title'][:70]}")

df['pred_ai'] = predictions
print(f"\nDone. Classified {len(predictions)} sessions.")


  0%|          | 0/231 [00:00<?, ?it/s]

✓ [AI] How to get data out of complex documents using AI
✓ [no] Everyone can be (and should be) a housing reporter
✓ [AI] AI concepts you should know
✓ [no] From delving into data to door knocking: Finding people behind the num
✓ [no] Collaborating in an intergenerational workspace: let's get to yes
✓ [no] Wanna get hired? Tales from the trenches
✓ [no] Immigration data and how to make sense of it
✓ [no] Utilizing new police misconduct and employment databases
✓ [no] Using data to find sources
✓ [no] Demo: The Gun Violence Data Hub
✓ [no] Connect with skeptical and disengaged audiences ... with data!
✓ [no] How to scrape and analyze sports data in R to tell stories that stand 
✓ [no] Customizing ggplot for yourself or your organization
✓ [no] Fun with hypotheticals: Investigating your dubious data dilemmas in re
✓ [AI] How to use AI responsibly
✓ [no] Challenges with cross-domain collaboration
✓ [AI] Enhancing public records work with LLMs
✓ [no] Data investigation frameworks: How to t

## Section 3: Accuracy, Precision, Recall, F1

In [4]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = df['label_ai'].astype(int)
y_pred = df['pred_ai'].astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print('Confusion matrix:')
print(f'  True Positives:  {tp}')
print(f'  True Negatives:  {tn}')
print(f'  False Positives: {fp}')
print(f'  False Negatives: {fn}')
print()

correct = (y_true == y_pred).sum()
total = len(y_true)
accuracy_pct = (correct / total) * 100

report = classification_report(
    y_true,
    y_pred,
    target_names=['Non-AI', 'AI'],
    output_dict=True,
    zero_division=0,
)

precision = report['AI']['precision']
recall = report['AI']['recall']

print(f'Accuracy (correct/total): {correct}/{total} = {accuracy_pct:.2f}%')
print(f'Precision (AI): {precision:.3f}')
print(f'Recall (AI): {recall:.3f}')

Confusion matrix:
  True Positives:  29
  True Negatives:  189
  False Positives: 12
  False Negatives: 1

Accuracy (correct/total): 218/231 = 94.37%
Precision (AI): 0.707
Recall (AI): 0.967


## Section 4: False Positives & False Negatives

In [5]:
cols = ['session_title', 'tracks', 'description_clean']

false_positives = df[(df['pred_ai'] == True) & (df['label_ai'] == False)][cols]
false_negatives = df[(df['pred_ai'] == False) & (df['label_ai'] == True)][cols]

print(f'=== FALSE POSITIVES ({len(false_positives)}) ===')
print('Model said AI, track said no:\n')
for _, row in false_positives.iterrows():
    print(f'  Title:  {row["session_title"]}')
    print(f'  Tracks: {row["tracks"]}')
    print(f'  Desc:   {row["description_clean"]}...')
    print()

print(f'=== FALSE NEGATIVES ({len(false_negatives)}) ===')
print('Model said not AI, track said yes:\n')
for _, row in false_negatives.iterrows():
    print(f'  Title:  {row["session_title"]}')
    print(f'  Tracks: {row["tracks"]}')
    print(f'  Desc:   {row["description_clean"]}...')
    print()

=== FALSE POSITIVES (12) ===
Model said AI, track said no:

  Title:  10 things you didn’t know you could do in R
  Tracks: ['Data analysis', 'Tools & Tech']
  Desc:   R is great for quick data exploration, visualization and spatial analysis. But did you know you can extract text from images, send customized emails and chat with PDFs all without ever leaving the cozy comfort of R?
In this session, we&#39;ll zip through a few lesser-known or relatively new R packages for working with large language models, media and databases. Though pitched at intermediate users interested in getting more mileage out of R, this demo will also give beginners a taste of what&#39;s possible. As a bonus, we&#39;ll invite attendees to complete a short survey sharing some of the unexpected ways they use R in their newsrooms and circulate the results in a tip sheet after the session.
This session is good for people with some experience working with R. Laptops will be provided....

  Title:  Tips to investigat